## Demonstration: Using the Provenance Extension

**Prerequisite to run this notebook:** having [almond jupyter plugin](https://almond.sh/) installed, and running cells using a Scala kernel

To use Spark, we can use `$ivy` magic: https://almond.sh/docs/usage-spark

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

We can also use `ivy` to load project source code from local Ivy/Maven cache (run `make publish-local` to update):

In [2]:
val version = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % version)  // use porogrammatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

version: String = "0.0.1"

Import libraries (Spark, dataprovenance project, ...):

In [3]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame}
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.functions._

import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension
import org.dataprov.dp.WhyProvenanceBuilder

import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame}
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.functions._
import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension
import org.dataprov.dp.WhyProvenanceBuilder

The WhyProvenanceBuilder is a simple implementation of the ProvenanceBuilder trait that captures 
the logical plan and physical plan for each transformation. 

It can be used to track the lineage of data through the transformations and understand how the final result was derived from the input data.

**Initialize Spark session:**
- The provenance builder can be customized to use different provenance models (e.g. WhyProvenanceBuilder)
- A new provenance builder can be created directly by overriding the operations (e.g. provType, single, join, distinct, aggregate)

In [4]:
// Create SparkSession
val sparkWhy = SparkSession.builder()
    .appName("notebook-demo-why-provenance")
    .master("local[*]")
    .withExtensions(
        // can be customized with different operators and builders
        new ProvenanceExtension(provenanceBuilder = WhyProvenanceBuilder)
    )
    .config("spark.provenance.enabled", "true")
    .getOrCreate()

println(s"Spark provenance enabled: ${sparkWhy.conf.get("spark.provenance.enabled")}")

// Set log level to ERROR to reduce verbosity
sparkWhy.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/01 14:16:04 INFO SparkContext: Running Spark version 4.1.1
26/06/01 14:16:04 INFO SparkContext: OS info Mac OS X, 26.4.1, aarch64
26/06/01 14:16:04 INFO SparkContext: Java version 17.0.10+7
26/06/01 14:16:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/01 14:16:04 INFO ResourceUtils: ==============================================================
26/06/01 14:16:04 INFO ResourceUtils: No custom resources configured for spark.driver.
26/06/01 14:16:04 INFO ResourceUtils: ==============================================================
26/06/01 14:16:04 INFO SparkContext: Submitted application: notebook-demo-why-provenance
26/06/01 14:16:04 INFO SecurityManager: Changing view acls to: mac-ABALLA16
26/06/01 14:16:04 INFO SecurityManager: Changing modify acls to: mac-ABALLA16
26/06/01 14:16:04 INFO SecurityManager: Changing

Spark provenance enabled: true


sparkWhy: SparkSession = org.apache.spark.sql.classic.SparkSession@6e4bf956

Create Spark dataframe:

In [5]:
val df: DataFrame = sparkWhy.createDataFrame(
    Seq(
        ("A", Date.valueOf("2026-01-15"), 10.0, 90),
        ("A", Date.valueOf("2026-01-16"), 10.0, 120),
        ("A", Date.valueOf("2026-01-17"), 5.0, 300),
        ("B", Date.valueOf("2026-01-15"), 100.0, 20),
        ("B", Date.valueOf("2026-01-16"), 100.0, 30),
        ("C", Date.valueOf("2026-01-17"), 80.0, 60)
    )
).toDF("product", "date", "price", "sales")

df.show()

+-------+----------+-----+-----+
|product|      date|price|sales|
+-------+----------+-----+-----+
|      A|2026-01-15| 10.0|   90|
|      A|2026-01-16| 10.0|  120|
|      A|2026-01-17|  5.0|  300|
|      B|2026-01-15|100.0|   20|
|      B|2026-01-16|100.0|   30|
|      C|2026-01-17| 80.0|   60|
+-------+----------+-----+-----+



df: DataFrame = [product: string, date: date ... 2 more fields]

**Add provenance column to the DataFrame**

The provenance column is added using the `addProvenanceColumn` method, which is provided by the `DataFrameProvenanceTransformations` trait. 

This method adds a new column to the DataFrame that contains the provenance information for each row. By default the provenance type is a uuid but can be customized by choosing a column name of the dataframe.

By adding the provenance column, users can easily trace back the origin of each row and understand the transformations that were applied to it, which can be useful for debugging and analysing purposes.


In [10]:
val dfWithProv : DataFrame = df.addProvenanceColumn
dfWithProv.show(false)

val dfWithProv2 : DataFrame = df.addProvenanceColumn(col("product"))
dfWithProv2.show(false)

+-------+----------+-----+-----+------------------------------------+
|product|date      |price|sales|_provenance_tag                     |
+-------+----------+-----+-----+------------------------------------+
|A      |2026-01-15|10.0 |90   |f54c4f8f-a18f-4aa9-83fb-4272fdd033ec|
|A      |2026-01-16|10.0 |120  |c564d936-53cb-40c7-b0de-203f843e675d|
|A      |2026-01-17|5.0  |300  |4ef57f9a-9ca9-4414-b5f2-02274d60a7c8|
|B      |2026-01-15|100.0|20   |ce7cf0fc-982a-4f2a-b7e7-df973be51d96|
|B      |2026-01-16|100.0|30   |05b5304e-c018-4e0b-a7ea-2a2211e60620|
|C      |2026-01-17|80.0 |60   |2c612370-b584-4120-b747-63dc0a9fe6d1|
+-------+----------+-----+-----+------------------------------------+

+-------+----------+-----+-----+---------------+
|product|date      |price|sales|_provenance_tag|
+-------+----------+-----+-----+---------------+
|A      |2026-01-15|10.0 |90   |A              |
|A      |2026-01-16|10.0 |120  |A              |
|A      |2026-01-17|5.0  |300  |A              |
|B   

dfWithProv: DataFrame = [product: string, date: date ... 3 more fields]
dfWithProv2: DataFrame = [product: string, date: date ... 3 more fields]

**How to use the new DataFrame with provenance to understand the provenance of the data and debug queries**

You can use the same SQL or DataFrame API as before, but now you have access to the provenance information for debugging and analysis purposes.
You have nothing more to do, the provenance column will be calculated automatically.

Example query: find products with price > 50 and sales < 50.

In [7]:
// SQL API with provenance
dfWithProv.createOrReplaceTempView("why_prov")
val resultWithProv = sparkWhy.sql(
    """
    SELECT product, date, price, sales
    FROM why_prov
    WHERE price > 50 AND sales < 50
    """
)
resultWithProv.show(false)

// DataFrame API with provenance
val resultWithProv2 = dfWithProv
    .filter(col("price") > 50 && col("sales") < 50)
    .select("product", "date", "price", "sales")
resultWithProv2.show(false)

+-------+----------+-----+-----+------------------------------------+
|product|date      |price|sales|_provenance_tag                     |
+-------+----------+-----+-----+------------------------------------+
|B      |2026-01-15|100.0|20   |f32e197e-254e-4657-92db-bbdcac0d8f9c|
|B      |2026-01-16|100.0|30   |fabbf113-9d9c-46eb-9d11-ee0cacb1d5ad|
+-------+----------+-----+-----+------------------------------------+

+-------+----------+-----+-----+------------------------------------+
|product|date      |price|sales|_provenance_tag                     |
+-------+----------+-----+-----+------------------------------------+
|B      |2026-01-15|100.0|20   |f32e197e-254e-4657-92db-bbdcac0d8f9c|
|B      |2026-01-16|100.0|30   |fabbf113-9d9c-46eb-9d11-ee0cacb1d5ad|
+-------+----------+-----+-----+------------------------------------+



resultWithProv: DataFrame = [product: string, date: date ... 3 more fields]
resultWithProv2: DataFrame = [product: string, date: date ... 3 more fields]